In [27]:
import json
import os
import random

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from itertools import combinations
from openai import OpenAI
from dotenv import load_dotenv

from utils.edinet_api import get_doc_name
from utils.ELO import EloRatingSystem, get_num_games
from utils.datapath import (
    edinet_codes_path,
    nikkei_225_path,
    docs_metadata_path,
    wins_signals_path,
    elo_signals_path,
)

In [28]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")

client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

In [29]:
df = pd.read_excel(edinet_codes_path)

with open(nikkei_225_path) as f:
    nikkei_225 = json.load(f)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [30]:
# TODO:
# Create a dictionary of quarters and document paths - DONE
# Create combinations of all documents in a quarter or random combinations - DONE
# Create winner function - DONE
# Create output dictionary of document -> number of wins (for all combinations) - DONE
# Create a dictionary of (doc1, doc2) -> winner for ELO score (for all combinations and random combinations) - DONE

# Test with random winner, and then with Grok API

In [35]:
# Create a dictionary of quarters and document paths - DONE

quarterly_docs = defaultdict(list)

for doc in docs_metadata:
    period_end_date = doc["periodEnd"]
    period_end_ts = pd.Timestamp(period_end_date)
    period_end_quater = f"{period_end_ts.year}-{period_end_ts.quarter}"
    directory_path = os.path.join("..", "documents", period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(directory_path, save_name)

    quarterly_docs[period_end_quater].append(output_path)

In [36]:
# Create combinations of all documents in a quarter or random combinations

quarterly_combinations = defaultdict(list)

for quarter, docs in quarterly_docs.items():
    quarterly_combinations[quarter] = list(combinations(docs, 2))
    print(quarter, len(docs), len(list(combinations(docs, 2))))

2022-4 193 18528
2023-1 35 595
2023-2 223 24753
2023-3 222 24531
2023-4 195 18915


In [37]:
def generate_random_pdf_pair(quarter, iterations):
    docs = quarterly_docs[quarter]

    for i in range(iterations):
        selected_pdfs = random.sample(docs, 2)

        yield (
            selected_pdfs[0],
            selected_pdfs[1],
        )

In [38]:
for quarter in quarterly_docs:
    print(quarter)
    for d1, d2 in generate_random_pdf_pair(quarter, 3):
        print(d1, d2)

2022-4
../documents/2022-4/E00436_味の素株式会社_140_S100Q42V.pdf ../documents/2022-4/E02126_三菱重工業株式会社_140_S100Q36U.pdf
../documents/2022-4/E00988_富士フイルムホールディングス株式会社_140_S100Q7W2.pdf ../documents/2022-4/E04762_オリックス株式会社_140_S100Q62F.pdf
../documents/2022-4/E01332_古河電気工業株式会社_140_S100Q54D.pdf ../documents/2022-4/E01630_テルモ株式会社_140_S100Q6KM.pdf
2023-1
../documents/2023-1/E01122_ＡＧＣ株式会社_140_S100QRLA.pdf ../documents/2023-1/E04999_トレンドマイクロ株式会社_140_S100QQQD.pdf
../documents/2023-1/E04999_トレンドマイクロ株式会社_140_S100QQQD.pdf ../documents/2023-1/E00395_キリンホールディングス株式会社_140_S100QPV6.pdf
../documents/2023-1/E25850_株式会社ネクソン_140_S100QQBG.pdf ../documents/2023-1/E01162_東海カーボン株式会社_140_S100QPEM.pdf
2023-2
../documents/2023-2/E27633_東急不動産ホールディングス株式会社_140_S100RL9W.pdf ../documents/2023-2/E02143_いすゞ自動車株式会社_140_S100RME5.pdf
../documents/2023-2/E05460_株式会社ディー・エヌ・エー_140_S100RNB7.pdf ../documents/2023-2/E02081_ルネサスエレクトロニクス株式会社_140_S100RHF7.pdf
../documents/2023-2/E00322_コムシスホールディングス株式会社_140_S100RKN8.pdf ../documents/2023-

In [39]:
from utils.llm_api import call_grok_api, prepare_prompt_messages
from utils.pdf import extract_pdf_text
from utils.data import nikkei_225, edinet_to_stock_code_map


def get_winner(pdf1_path, pdf2_path):
    return random.choice([0, 1])  #


def get_winner_grok(pdf1_path, pdf2_path):
    pdf1_txt, pdf2_txt = extract_pdf_text(pdf1_path), extract_pdf_text(pdf2_path)
    completion = call_grok_api(client, prepare_prompt_messages(pdf1_txt, pdf2_txt))
    response = json.loads(completion.choices[0].message.content)
    return response["winner"]


def get_edinet_code_from_path(pdf_path):
    """
    path = f"../documents/{quarter}/{edinet_code}_{filer}_{doc_type_code}_{doc_id}.{FILE_EXT}"
    """
    return pdf_path.split("/")[-1].split("_")[0]

def get_stock_code_from_path(pdf_path):
    """
    path = f"../documents/{quarter}/{edinet_code}_{filer}_{doc_type_code}_{doc_id}.{FILE_EXT}"
    """
    return edinet_to_stock_code_map.get(get_edinet_code_from_path(pdf_path))

In [40]:
# Test get_code_from_path()
for quarter, combinations in quarterly_combinations.items():
    for pdf1_path, pdf2_path in combinations:
        codes = [get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        print(codes)
        break

['6506', '7453']
['7453', '9983']
['1928', '6506']
['1928', '6506']
['1928', '3086']


In [41]:
# Document winners
quarterly_wins = defaultdict(lambda: defaultdict(int))

for quarter, combinations in quarterly_combinations.items():
    for pdf1_path, pdf2_path in combinations:
        winner = get_winner(pdf1_path, pdf2_path)
        codes = [get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        winner_name = codes[winner - 1]
        quarterly_wins[quarter][winner_name] += 1

In [42]:
# Battle outcomes
quarterly_battle_outcomes = defaultdict(list)

for quarter in quarterly_docs:
    for pdf1_path, pdf2_path in generate_random_pdf_pair(quarter, 2000):
        winner = get_winner(pdf1_path, pdf2_path)
        codes = [get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        quarterly_battle_outcomes[quarter].append(tuple(codes + [winner]))

In [43]:
# Create quarterly signals from quarterly wins
quarterly_signals_wins = {}

for quarter, docs in quarterly_docs.items():
    num_docs = len(docs)
    quarterly_signals_wins[quarter] = {
        code: wins / num_docs for code, wins in quarterly_wins[quarter].items()
    }

In [44]:

with open(wins_signals_path, 'w', encoding='utf-8') as f:
    json.dump(quarterly_signals_wins, f)

In [45]:
# ELO carried over quarters!

quarterly_signals_elo = {}
elo_system = EloRatingSystem()

In [46]:
# Battle outcomes

for quarter, docs in quarterly_docs.items():
    for pdf1_path, pdf2_path in generate_random_pdf_pair(
        quarter, get_num_games(len(docs))
    ):
        winner = get_winner(pdf1_path, pdf2_path)
        companyA_code, companyB_code = [
            get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
        ]
        elo_system.update_ratings(companyA_code, companyB_code, winner)

    quarterly_signals_elo[quarter] = elo_system.get_all_ratings()

In [47]:
with open(elo_signals_path, 'w', encoding='utf-8') as f:
    json.dump(quarterly_signals_elo, f)

In [ ]:
# The signal needs a date!